# MVRV Exit Strategy

**The Thesis:** 
- Enter on SOPR capitulation (fear/undervaluation)
- Exit on MVRV elevation (greed/overvaluation)

**MVRV = Market Value / Realized Value**
- MVRV < 1 = Market cap below cost basis = undervalued
- MVRV > 2 = Market cap 2x cost basis = getting expensive
- MVRV > 3 = Historically marks cycle tops

This is a valuation-based exit vs our previous attempts (price-based, sentiment-based).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("MVRV Exit Strategy 📊")

In [ ]:
# Load all data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
mvrv_z = pd.read_parquet(DATA_DIR / "mvrv_z.parquet").rename(columns={"value": "mvrv_z"}).set_index("time")
mvrv_sth = pd.read_parquet(DATA_DIR / "mvrv_sth.parquet").rename(columns={"value": "mvrv_sth"}).set_index("time")

# Merge all
df = sopr.join(sopr_sth, how='inner').join(price, how='inner')
df = df.join(mvrv, how='inner').join(mvrv_z, how='inner').join(mvrv_sth, how='inner')
df = df.sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Explore MVRV
print("MVRV Statistics:")
print(f"  Min: {df['mvrv'].min():.2f}")
print(f"  Max: {df['mvrv'].max():.2f}")
print(f"  Median: {df['mvrv'].median():.2f}")
print(f"  Current: {df['mvrv'].iloc[-1]:.2f}")
print(f"\nPercentiles:")
for p in [25, 50, 75, 90, 95]:
    print(f"  {p}th: {df['mvrv'].quantile(p/100):.2f}")

print(f"\nMVRV Z-Score Statistics:")
print(f"  Min: {df['mvrv_z'].min():.2f}")
print(f"  Max: {df['mvrv_z'].max():.2f}")
print(f"  Median: {df['mvrv_z'].median():.2f}")
print(f"  Current: {df['mvrv_z'].iloc[-1]:.2f}")

In [ ]:
# Visualize MVRV with price
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=['BTC Price', 'MVRV', 'MVRV Z-Score'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV'), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='green', row=2, col=1)
fig.add_hline(y=2, line_dash='dot', line_color='orange', row=2, col=1)
fig.add_hline(y=3, line_dash='dot', line_color='red', row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv_z'], name='MVRV Z'), row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=3, col=1)
fig.add_hline(y=2, line_dash='dot', line_color='orange', row=3, col=1)
fig.add_hline(y=4, line_dash='dot', line_color='red', row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=700, title_text='MVRV as Exit Indicator')
fig.show()

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")

---
## MVRV Exit Backtester

In [ ]:
def backtest_mvrv_exit(
    df: pd.DataFrame,
    entries: pd.Series,
    exit_mvrv: float = 2.0,
    exit_mvrv_z: float = None,  # Alternative: use Z-score
    stop_loss: float = None,
    max_hold_days: int = 365,  # Longer hold for valuation-based
):
    """Backtest with MVRV-based exit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        entry_mvrv = df['mvrv'].iloc[entry_idx]
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            current_mvrv_z = df['mvrv_z'].iloc[j]
            days_held = j - entry_idx
            
            # Check stop loss
            if stop_loss is not None:
                pnl = (current_price - entry_price) / entry_price
                if pnl <= -stop_loss:
                    exit_date = current_date
                    exit_price = entry_price * (1 - stop_loss)
                    exit_reason = 'stop_loss'
                    break
            
            # Check MVRV exit
            mvrv_exit = current_mvrv >= exit_mvrv if exit_mvrv else False
            mvrv_z_exit = current_mvrv_z >= exit_mvrv_z if exit_mvrv_z else False
            
            if mvrv_exit or mvrv_z_exit:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'mvrv_overvalued' if mvrv_exit else 'mvrv_z_overvalued'
                break
            
            # Max hold
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'entry_mvrv': entry_mvrv,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'exit_mvrv': df.loc[exit_date, 'mvrv'] if exit_date in df.index else np.nan,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        # Skip entries during trade
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
def calc_stats(trades):
    if len(trades) == 0:
        return {'n_trades': 0, 'total_return': 0, 'win_rate': 0, 'profit_factor': 0, 'avg_days': 0}
    
    total_return = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
    avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
    
    gross_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].sum()
    gross_loss = abs(trades[trades['pnl_pct'] <= 0]['pnl_pct'].sum())
    profit_factor = gross_win / gross_loss if gross_loss > 0 else np.inf
    
    return {
        'n_trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'profit_factor': profit_factor,
        'avg_days': trades['days_held'].mean()
    }

---
## Test MVRV Exit Thresholds

In [ ]:
# Test various MVRV exit levels
mvrv_thresholds = [
    {'name': 'MVRV > 1.5', 'mvrv': 1.5, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV > 1.75', 'mvrv': 1.75, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV > 2.0', 'mvrv': 2.0, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV > 2.25', 'mvrv': 2.25, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV > 2.5', 'mvrv': 2.5, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV > 3.0', 'mvrv': 3.0, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV Z > 1', 'mvrv': 99, 'mvrv_z': 1, 'sl': None},
    {'name': 'MVRV Z > 2', 'mvrv': 99, 'mvrv_z': 2, 'sl': None},
    {'name': 'MVRV Z > 3', 'mvrv': 99, 'mvrv_z': 3, 'sl': None},
]

results = []

for config in mvrv_thresholds:
    trades = backtest_mvrv_exit(
        df=df,
        entries=entries,
        exit_mvrv=config['mvrv'],
        exit_mvrv_z=config['mvrv_z'],
        stop_loss=config['sl'],
        max_hold_days=365
    )
    
    stats = calc_stats(trades)
    stats['name'] = config['name']
    
    if len(trades) > 0:
        stats['mvrv_exits'] = trades['exit_reason'].str.contains('mvrv').sum()
        stats['max_hold_exits'] = (trades['exit_reason'] == 'max_hold').sum()
    else:
        stats['mvrv_exits'] = 0
        stats['max_hold_exits'] = 0
    
    results.append(stats)

results_df = pd.DataFrame(results)

print("MVRV EXIT THRESHOLDS (No Stop Loss)")
print("="*110)
print(f"{'Exit Rule':<20} {'Trades':>7} {'Return':>10} {'Win%':>7} {'AvgWin':>8} {'AvgLoss':>8} {'PF':>6} {'Days':>6} {'MVRV':>6} {'MaxH':>6}")
print("-"*110)
for _, row in results_df.iterrows():
    print(f"{row['name']:<20} {row['n_trades']:>7} {row['total_return']*100:>9.0f}% {row['win_rate']*100:>6.0f}% "
          f"{row['avg_win']*100:>7.1f}% {row['avg_loss']*100:>7.1f}% {row['profit_factor']:>6.2f} "
          f"{row['avg_days']:>6.0f} {row['mvrv_exits']:>6} {row['max_hold_exits']:>6}")

In [ ]:
# Now with stop loss backup
print("\n\nMVRV EXIT + 20% STOP LOSS")
print("="*110)

results_sl = []

for config in mvrv_thresholds:
    trades = backtest_mvrv_exit(
        df=df,
        entries=entries,
        exit_mvrv=config['mvrv'],
        exit_mvrv_z=config['mvrv_z'],
        stop_loss=0.20,
        max_hold_days=365
    )
    
    stats = calc_stats(trades)
    stats['name'] = config['name']
    
    if len(trades) > 0:
        stats['mvrv_exits'] = trades['exit_reason'].str.contains('mvrv').sum()
        stats['stop_exits'] = (trades['exit_reason'] == 'stop_loss').sum()
        stats['max_hold_exits'] = (trades['exit_reason'] == 'max_hold').sum()
    else:
        stats['mvrv_exits'] = 0
        stats['stop_exits'] = 0
        stats['max_hold_exits'] = 0
    
    results_sl.append(stats)

results_sl_df = pd.DataFrame(results_sl)

print(f"{'Exit Rule':<20} {'Trades':>7} {'Return':>10} {'Win%':>7} {'PF':>6} {'Days':>6} {'MVRV':>6} {'Stop':>6} {'MaxH':>6}")
print("-"*110)
for _, row in results_sl_df.iterrows():
    print(f"{row['name']:<20} {row['n_trades']:>7} {row['total_return']*100:>9.0f}% {row['win_rate']*100:>6.0f}% "
          f"{row['profit_factor']:>6.2f} {row['avg_days']:>6.0f} "
          f"{row['mvrv_exits']:>6} {row['stop_exits']:>6} {row['max_hold_exits']:>6}")

---
## Walk-Forward Validation

In [ ]:
def walk_forward_mvrv(df, entries, exit_mvrv, exit_mvrv_z, stop_loss, 
                       train_days=365, test_days=90, step_days=90):
    """Walk-forward for MVRV exit."""
    wf_results = []
    close = df['price']
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        if test_end <= test_start:
            break
        
        test_df = df.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        trades = backtest_mvrv_exit(
            df=test_df,
            entries=test_entries,
            exit_mvrv=exit_mvrv,
            exit_mvrv_z=exit_mvrv_z,
            stop_loss=stop_loss,
            max_hold_days=365
        )
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        n_trades = len(trades)
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        wf_results.append({
            'fold': fold,
            'period': df.index[test_start].strftime('%Y-%m'),
            'n_trades': n_trades,
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    return pd.DataFrame(wf_results)

In [ ]:
# Walk-forward test all strategies
strategies = [
    {'name': 'MVRV > 1.5 + 20% SL', 'mvrv': 1.5, 'mvrv_z': None, 'sl': 0.20},
    {'name': 'MVRV > 1.75 + 20% SL', 'mvrv': 1.75, 'mvrv_z': None, 'sl': 0.20},
    {'name': 'MVRV > 2.0 + 20% SL', 'mvrv': 2.0, 'mvrv_z': None, 'sl': 0.20},
    {'name': 'MVRV > 2.25 + 20% SL', 'mvrv': 2.25, 'mvrv_z': None, 'sl': 0.20},
    {'name': 'MVRV > 2.5 + 20% SL', 'mvrv': 2.5, 'mvrv_z': None, 'sl': 0.20},
    {'name': 'MVRV > 3.0 + 20% SL', 'mvrv': 3.0, 'mvrv_z': None, 'sl': 0.20},
    {'name': 'MVRV Z > 2 + 20% SL', 'mvrv': 99, 'mvrv_z': 2, 'sl': 0.20},
    {'name': 'MVRV Z > 3 + 20% SL', 'mvrv': 99, 'mvrv_z': 3, 'sl': 0.20},
    {'name': 'MVRV > 2.0 (no SL)', 'mvrv': 2.0, 'mvrv_z': None, 'sl': None},
    {'name': 'MVRV > 2.5 (no SL)', 'mvrv': 2.5, 'mvrv_z': None, 'sl': None},
]

wf_comparison = []

for strat in strategies:
    wf = walk_forward_mvrv(
        df, entries,
        exit_mvrv=strat['mvrv'],
        exit_mvrv_z=strat['mvrv_z'],
        stop_loss=strat['sl']
    )
    
    wf_comparison.append({
        'strategy': strat['name'],
        'beat_rate': wf['beat_hold'].mean(),
        'avg_excess': wf['excess'].mean(),
        'avg_trades': wf['n_trades'].mean()
    })

wf_comp_df = pd.DataFrame(wf_comparison).sort_values('beat_rate', ascending=False)

print("\nWALK-FORWARD COMPARISON - MVRV EXIT")
print("="*80)
print(f"{'Strategy':<30} {'Beat Rate':>12} {'Avg Excess':>12} {'Avg Trades':>12}")
print("-"*80)
for _, row in wf_comp_df.iterrows():
    print(f"{row['strategy']:<30} {row['beat_rate']*100:>11.0f}% {row['avg_excess']*100:>+11.1f}% {row['avg_trades']:>12.1f}")

In [ ]:
# Visualize
fig = go.Figure()

colors = ['green' if x > 0.55 else 'orange' if x > 0.50 else 'red' for x in wf_comp_df['beat_rate']]

fig.add_trace(go.Bar(
    x=wf_comp_df['strategy'],
    y=wf_comp_df['beat_rate'] * 100,
    marker_color=colors,
    text=[f"{x:.0f}%" for x in wf_comp_df['beat_rate']*100],
    textposition='outside'
))

fig.add_hline(y=50, line_dash='dash', line_color='red')
fig.add_hline(y=54, line_dash='dot', line_color='orange', annotation_text='54% baseline')

fig.update_layout(
    title='Walk-Forward Beat Rate - MVRV Exit',
    yaxis_title='Beat Buy & Hold %',
    xaxis_tickangle=-45,
    height=500
)
fig.show()

---
## Detailed Trade Analysis for Best Strategy

In [ ]:
# Get best strategy
best = wf_comp_df.iloc[0]
print(f"Best Strategy: {best['strategy']}")
print(f"Beat Rate: {best['beat_rate']*100:.0f}%")

# Get trades for best
# Parse the best strategy parameters
best_strat = strategies[[s['name'] for s in strategies].index(best['strategy'])]

best_trades = backtest_mvrv_exit(
    df=df,
    entries=entries,
    exit_mvrv=best_strat['mvrv'],
    exit_mvrv_z=best_strat['mvrv_z'],
    stop_loss=best_strat['sl'],
    max_hold_days=365
)

print(f"\nTrade Details:")
display_trades = best_trades.copy()
display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
display_trades['entry_price'] = display_trades['entry_price'].round(0).astype(int)
display_trades['exit_price'] = display_trades['exit_price'].round(0).astype(int)
display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)
display_trades['entry_mvrv'] = display_trades['entry_mvrv'].round(2)
display_trades['exit_mvrv'] = display_trades['exit_mvrv'].round(2)

print(display_trades.to_string(index=False))

In [ ]:
# Exit reason breakdown
print("\nEXIT REASON BREAKDOWN")
print("="*50)
print(best_trades['exit_reason'].value_counts())

print("\nWIN RATE BY EXIT REASON")
print("-"*50)
for reason in best_trades['exit_reason'].unique():
    subset = best_trades[best_trades['exit_reason'] == reason]
    win_rate = (subset['pnl_pct'] > 0).mean()
    avg_pnl = subset['pnl_pct'].mean()
    print(f"{reason}: {len(subset)} trades, {win_rate*100:.0f}% win, {avg_pnl*100:.1f}% avg")

In [ ]:
# Plot entries/exits on MVRV chart
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                    subplot_titles=['Price with Trades', 'MVRV'])

# Price
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price',
                         line=dict(color='blue', width=1)), row=1, col=1)

# Entry points
fig.add_trace(go.Scatter(
    x=pd.to_datetime(best_trades['entry_date']),
    y=best_trades['entry_price'],
    mode='markers',
    marker=dict(symbol='triangle-up', size=12, color='green'),
    name='Entry'
), row=1, col=1)

# Exit points
colors = ['green' if p > 0 else 'red' for p in best_trades['pnl_pct']]
fig.add_trace(go.Scatter(
    x=pd.to_datetime(best_trades['exit_date']),
    y=best_trades['exit_price'],
    mode='markers',
    marker=dict(symbol='triangle-down', size=12, color=colors),
    name='Exit'
), row=1, col=1)

# MVRV
fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV',
                         line=dict(color='purple', width=1)), row=2, col=1)

# MVRV threshold
fig.add_hline(y=best_strat['mvrv'] if best_strat['mvrv'] < 10 else 2, 
              line_dash='dash', line_color='red', row=2, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=700, title_text=f"Trades: {best['strategy']}")
fig.show()

---
## Summary

In [ ]:
print("\n" + "="*70)
print("MVRV EXIT STRATEGY SUMMARY")
print("="*70)

print(f"\n🏆 BEST MVRV EXIT: {best['strategy']}")
print(f"   Walk-Forward Beat Rate: {best['beat_rate']*100:.0f}%")
print(f"   Avg Excess Return: {best['avg_excess']*100:+.1f}%")

print(f"\n📊 vs BASELINE (54% trailing stop)")
improvement = best['beat_rate'] - 0.54
print(f"   Improvement: {improvement*100:+.0f}%")

if best['beat_rate'] > 0.60:
    verdict = "✅ SIGNIFICANT IMPROVEMENT!"
elif best['beat_rate'] > 0.55:
    verdict = "✅ MODEST IMPROVEMENT"
elif best['beat_rate'] > 0.50:
    verdict = "⚠️ MARGINAL (similar to baseline)"
else:
    verdict = "❌ NO IMPROVEMENT"

print(f"\n🎯 VERDICT: {verdict}")
print("\n" + "="*70)

In [ ]:
# Save results
import json

mvrv_results = {
    'strategy': 'mvrv_exit',
    'entry': 'SOPR < 1 AND STH_SOPR < 1',
    'exit': 'MVRV > threshold OR stop_loss',
    'walk_forward_comparison': wf_comp_df.to_dict('records'),
    'best_strategy': {
        'name': best['strategy'],
        'beat_rate': float(best['beat_rate']),
        'avg_excess': float(best['avg_excess'])
    },
    'in_sample_results': results_sl_df.to_dict('records')
}

with open('../data/sopr_mvrv_exit_results.json', 'w') as f:
    json.dump(mvrv_results, f, indent=2, default=str)

print("Saved to ../data/sopr_mvrv_exit_results.json")